In [ ]:
"""
Script simples para NER via LLM com:
- timeout de 90s por chamada (mata a requisição e segue);
- cache/retomada via JSONL (predições e erros);
- política: preencher 500 amostras válidas; se alguma falhar, pega a próxima (501, 502, ...).

Entradas esperadas:
- Um JSONL com linhas no formato: {"id": <opcional>, "tokens": ["João", "foi", "a", "Roma", "."]}
  Se 'id' não existir, o script usa o índice da linha como ID estável.

Requisitos:
- requests
- Python 3.8+
"""

import argparse
import json
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import requests


In [ ]:
DEFAULT_HOST = "http://127.0.0.1:11434"        # Ollama
DEFAULT_MODEL = "gemma2:9b"                    # use o que você tiver local
DEFAULT_N_SAMPLES = 500
TIMEOUT_SECONDS = 90

In [ ]:
def sanitize(s: str) -> str:
    return s.replace("/", "_").replace(":", "_").replace(" ", "_")

def preds_path_for(model: str) -> Path:
    return Path(f"ner_preds_{sanitize(model)}.jsonl")

def errs_path_for(model: str) -> Path:
    return Path(f"ner_errors_{sanitize(model)}.jsonl")

In [ ]:
# =====================
def read_jsonl(path: Path) -> List[Dict]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def append_jsonl(path: Path, obj: Dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def load_dataset_jsonl(path: Path) -> List[Dict]:
    data = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if not line.strip():
                continue
            row = json.loads(line)
            # Se não tiver id, usamos o índice da linha (estável se arquivo não muda)
            row_id = row.get("id", i)
            tokens = row["tokens"]
            data.append({"id": row_id, "tokens": tokens})
    return data

In [ ]:
iob_labels = (
    "O "
    "B-baciaSedimentar I-baciaSedimentar "
    "B-epoca I-epoca "
    "B-idade I-idade "
    "B-periodo I-periodo "
    "B-eon I-eon "
    "B-era I-era "
    "B-magmaticas I-magmaticas "
    "B-metamorficas I-metamorficas "
    "B-sedimentaresSiliciclasticas I-sedimentaresSiliciclasticas "
    "B-sedimentaresCarbonaticas I-sedimentaresCarbonaticas "
    "B-unidadeEstratigrafica I-unidadeEstratigrafica "
    "B-contextoGeologicoDeBacia I-contextoGeologicoDeBacia "
    "B-ambienteSedimentacao I-ambienteSedimentacao "
    "B-constituinteRochaSedimentar I-constituinteRochaSedimentar "
    "B-fosseis I-fosseis "
    "B-planctonico I-planctonico "
    "B-bentonico I-bentonico "
    "B-mineral I-mineral "
    "B-procedimentoMetodologico I-procedimentoMetodologico"
)

def build_prompt(
    tokens: List[str],
    few_shot_block: Optional[str] = None,  # << novo: bloco já formatado
    tokens_to_text=lambda toks: json.dumps(toks, ensure_ascii=False),
) -> str:
    N = len(tokens)
    header = f"""
Você é um anotador especialista em NER. Rotule **cada token** usando o esquema **IOB2**.

Regras (case-sensitive):
1) Use apenas os rótulos:
{iob_labels}

2) IOB2 (cheque silenciosamente):
• Uma entidade inicia em B-<TIPO>.
• I-<TIPO> só após B-<TIPO> ou I-<TIPO> do mesmo TIPO.
• Nunca iniciar com I-.

3) Formato ÚNICO de saída (sem texto extra):
• Uma única linha com N pares índice-rótulo (1-based), separados por espaço.
• Cada par: <índice>-<RÓTULO>.
• Ex.: 1-O 2-B-magmaticas 3-I-magmaticas ... N-O

4) Se houver N tokens, produza exatamente N rótulos.
""".strip()

    examples = f"\n\n### Exemplos\n{few_shot_block}" if few_shot_block else ""

    task = f"""
### Tarefa
Tokens: {tokens_to_text(tokens)}

### Saída esperada
Retorne **apenas**: {{"tags": ["O", "B-...", "..."]}} com {N} itens.
""".strip()

    return f"{header}{examples}\n\n{task}"

In [ ]:
def parse_llm_labels_text(text: str) -> Optional[List[str]]:
    # 1) tenta JSON raiz com "tags" ou "labels"
    try:
        obj = json.loads(text)
        labels = obj.get("tags") or obj.get("labels")
        if isinstance(labels, list) and all(isinstance(x, str) for x in labels):
            return labels
    except Exception:
        pass

    # 2) fallback: tenta extrair a lista JSON dentro do texto
    try:
        start = text.index("[")
        end = text.rindex("]") + 1
        inner = text[start:end]
        labels = json.loads(inner)
        if isinstance(labels, list) and all(isinstance(x, str) for x in labels):
            return labels
    except Exception:
        pass

    return None

In [ ]:
def call_ollama_generate(host: str, model: str, prompt: str, timeout_s: int) -> str:
    """
    Usa /api/generate no modo não-streaming com timeout total.
    Se exceder timeout, levanta requests.Timeout (tratado acima).
    """
    url = f"{host.rstrip('/')}/api/generate"
    payload = {"model": model, "prompt": prompt, "stream": False}
    # Comentário: timeout=TIMEOUT_SECONDS garante que a conexão seja encerrada se passar do limite.
    r = requests.post(url, json=payload, timeout=timeout_s)
    r.raise_for_status()
    data = r.json()
    # Ollama retorna 'response' com o texto completo
    return data.get("response", "")


In [ ]:
def build_candidate_queues(n_total: int, n_primary: int, seed: int) -> Tuple[List[int], List[int]]:
    rng = random.Random(seed)
    idxs = list(range(n_total))
    rng.shuffle(idxs)
    primary = idxs[:n_primary]
    backup = idxs[n_primary:]
    return primary, backup

def already_done_ids(preds_file: Path, errs_file: Path) -> Tuple[set, set]:
    done, failed = set(), set()
    for row in read_jsonl(preds_file):
        if "id" in row:
            done.add(row["id"])
    for row in read_jsonl(errs_file):
        if "id" in row:
            failed.add(row["id"])
    return done, failed


In [ ]:
def select_few_shot(exemplars: Dataset, k=FEW_SHOT_K) -> list[dict]:
    return random.sample(list(exemplars), k)

def _format_few_shot_block(few_shot):
    if isinstance(few_shot, str):
        return few_shot.strip()[:4000]

    parts = []
    try:
        for ex in few_shot:
            toks = ex.get("tokens") if isinstance(ex, dict) else ex["tokens"]
            tags = ex.get("ner_tags") if isinstance(ex, dict) else ex["ner_tags"]
            parts.append(json.dumps({"tokens": toks, "tags": tags}, ensure_ascii=False))
    except Exception:
        return str(few_shot)[:4000]

    txt = "Exemplos rotulados (IOB2):\n" + "\n".join(parts)
    return txt[:4000]

In [ ]:
def run_ner(
    data: List[Dict],
    host: str,
    model: str,
    n_samples: int,
    seed: int = 42,
    label_set: Optional[List[str]] = None,  # pode ficar, mesmo sem uso direto no prompt
    few_shot: Optional[List] = None,
    fmt_example=lambda ex: ex,
    tokens_to_text=lambda toks: json.dumps(toks, ensure_ascii=False),
) -> None:
    preds_file = preds_path_for(model)
    errs_file = errs_path_for(model)

    done_ids, failed_ids = already_done_ids(preds_file, errs_file)

    # Filtra dataset para IDs ainda não processados (nem sucesso, nem falha)
    indexed = list(enumerate(data))  # (idx, {"id":..., "tokens":[...]})
    remaining = [(i, row) for i, row in indexed if row["id"] not in done_ids]

    if not remaining:
        print("Nada a fazer: todas as amostras deste dataset já foram processadas (sucesso).")
        return

    # Constrói filas: 500 primeiros e backups
    primary, backup = build_candidate_queues(len(remaining), n_samples, seed)

    successes = 0
    needed = n_samples

    # fila de trabalho começa com os 500 primeiros índices (posições dentro de 'remaining')
    work_queue: List[int] = list(primary)
    backup_iter = iter(backup)

    started_at = time.time()
    print(f"Iniciando - alvo de {n_samples} amostras válidas. Modelo={model} Host={host}")

    while work_queue and successes < needed:
        pos = work_queue.pop(0)
        i, row = remaining[pos]
        sample_id = row["id"]
        if sample_id in done_ids:
            continue  # já salvo em execuções anteriores

        tokens = row["tokens"]
        prompt = build_prompt(
            tokens,
            few_shot=few_shot,
            fmt_example=fmt_example,
            tokens_to_text=tokens_to_text,
        )

        t0 = time.time()
        try:
            text = call_ollama_generate(host, model, prompt, TIMEOUT_SECONDS)
            labels = parse_llm_labels_text(text)

            # Validação básica
            if not labels or len(labels) != len(tokens):
                raise ValueError("Resposta inválida da LLM (labels ausentes ou tamanho incorreto).")

            # Salva predição em JSONL (append-only)
            append_jsonl(
                preds_file,
                {
                    "id": sample_id,
                    "model": model,
                    "tokens": tokens,
                    "labels": labels,
                    "elapsed_s": round(time.time() - t0, 3),
                },
            )
            done_ids.add(sample_id)
            successes += 1

        except (requests.Timeout, requests.ConnectionError) as e:
            # Timeout ou conexão: registra falha e enfileira um backup
            append_jsonl(
                errs_file,
                {
                    "id": sample_id,
                    "model": model,
                    "error": "timeout_or_connection",
                    "msg": str(e),
                    "elapsed_s": round(time.time() - t0, 3),
                },
            )
            failed_ids.add(sample_id)
            # Comentário: puxa um próximo backup para manter a meta de 500.
            try:
                nxt = next(backup_iter)
                work_queue.append(nxt)
            except StopIteration:
                pass  # sem mais backups; seguimos até esvaziar a fila

        except Exception as e:
            # Qualquer erro de parse/validação/etc: registra falha e puxa backup
            append_jsonl(
                errs_file,
                {
                    "id": sample_id,
                    "model": model,
                    "error": "inference_or_parse_error",
                    "msg": str(e),
                    "elapsed_s": round(time.time() - t0, 3),
                },
            )
            failed_ids.add(sample_id)
            try:
                nxt = next(backup_iter)
                work_queue.append(nxt)
            except StopIteration:
                pass

        # Log leve a cada 25 sucessos
        if successes % 25 == 0 and successes > 0:
            print(f"[{successes}/{needed}] ok — tempo total {round(time.time() - started_at, 1)}s")

    print(
        f"Concluído: {successes} amostras salvas em {preds_file.name}. "
        f"Falhas acumuladas: {len(failed_ids)} (ver {errs_file.name})."
    )
    if successes < needed:
        print("Aviso: não foi possível atingir 500 válidas (dataset/LLM limitantes). Retome depois; o cache mantém o progresso.")


In [ ]:
few = select_few_shot(split_base["train"], FEW_SHOT_K)
few_block = _format_few_shot_block(few)

In [ ]:
run_ner(
    data,
    host=host,
    model=model,
    n_samples=500,
    seed=42,
    few_shot_block=few_block,   # << pronto
)